# Architecture bake-off — HAM10000 (Kaggle GPU)

Before spending the HPO budget, decide the **backbone**. This trains a short, fair
proxy run of several torchvision architectures on the *same* lesion-grouped split and
imbalance recipe as `kaggle_training.ipynb`, then ranks them on three axes that matter
for this project:

- **Balanced accuracy** (mean per-class recall, the promotion metric),
- **Calibration** (ECE — central to the uncertainty story),
- **Robustness** under darken / noise / blur (the documented failure mode).

It runs at **proxy fidelity** (256 px, a 40% stratified subset, few epochs) so each
model finishes in minutes — we only need the *ranking*, not final accuracy. The winner
then gets the full-fidelity HPO + training run.

**How to run:** Kaggle → *Add Input* → *Skin Cancer MNIST: HAM10000* (by `kmader`);
Settings → Accelerator → **GPU T4 x2**; Run All. Outputs land in
`/kaggle/working/results/bakeoff_results.json`.

## 1. Environment

In [ ]:
# Kaggle ships torch, torchvision, scikit-learn, matplotlib, pillow — nothing to install.
import os, json, time, csv, random, gc
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

IN_KAGGLE = Path("/kaggle/input").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
USE_AMP = device.type == "cuda"

# AMP helpers, tolerant of the torch.amp (>=2.3) vs torch.cuda.amp API split.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(): return _GradScaler(device.type, enabled=USE_AMP)
    def amp_ctx(): return _autocast(device.type, enabled=USE_AMP)
except Exception:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(): return _GradScaler(enabled=USE_AMP)
    def amp_ctx(): return _autocast(enabled=USE_AMP)

print(f"PyTorch {torch.__version__}  device: {device}  AMP: {USE_AMP}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    if supported and arch not in supported:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(0)} ({arch}) unsupported by this PyTorch build "
            f"(supports {supported}). Switch Settings -> Accelerator -> GPU T4 x2 and Run All.")
elif IN_KAGGLE:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again.")


## 2. Configuration

In [ ]:
# --- candidate backbones (all torchvision-native, so the winner is drop-in loadable
#     by skin/03_app.py via skin_model.load_trained_model with no extra deps) --------
ARCHITECTURES = [
    "efficientnet_v2_s",   # incumbent: modern conv net (~21M), strong on small data
    "convnext_small",      # modern conv net (~50M), keeps conv inductive bias
    "swin_v2_s",           # hierarchical vision transformer w/ local windows (~50M)
]
# ConvNeXt-V2 and DINOv2 live in `timm`, not torchvision; adopting either would need a
# small loader change in skin_model — a separate follow-up if a transformer/SSL backbone
# wins this round. Add more torchvision names above to widen the comparison.

# --- proxy fidelity (fast, ranking-only) ------------------------------------------
PROXY_IMG_SIZE  = 256    # common resolution all three accept (Swin-V2 is 256-native)
SUBSET_FRACTION = 0.40   # share of TRAIN kept, sampled PER CLASS so the class prior
                         # (= the logit-adjustment target) is preserved exactly
HEAD_EPOCHS = 2
FT_EPOCHS   = 6
HEAD_LR     = 1e-3
FT_LR       = 1e-4
BATCH_SIZE  = 32
LOGIT_ADJUST_TAU = 1.0   # keep the promoted recipe's imbalance corrector on, so the
                         # ranking reflects how these models will actually be trained

# --- split / data (same knobs as kaggle_training.ipynb) ---------------------------
SEED = 42
VAL_FRACTION = 0.15
VAL_CAP      = 200       # cap val images per class for a clean, comparable metric
TRAIN_CAP    = 100_000
NUM_WORKERS  = min(4, os.cpu_count() or 2)
VAL_RESIZE   = round(PROXY_IMG_SIZE * 256 / 224)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if IN_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/skin-cancer-mnist-ham10000")
    OUT_DIR   = Path("/kaggle/working/results")
else:
    INPUT_DIR = Path.cwd() / "data" / "ham10000_raw"
    OUT_DIR   = Path.cwd() / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_TO_CLASS = {
    "akiec": "actinic_keratoses", "bcc": "basal_cell_carcinoma",
    "bkl": "benign_keratosis-like_lesions", "df": "dermatofibroma",
    "nv": "melanocytic_nevi", "mel": "melanoma", "vasc": "vascular_lesions",
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Candidates:", ", ".join(ARCHITECTURES))
print(f"Proxy: {PROXY_IMG_SIZE}px  subset {SUBSET_FRACTION:.0%}  "
      f"head {HEAD_EPOCHS}ep + ft {FT_EPOCHS}ep  tau={LOGIT_ADJUST_TAU}")


## 3. Lesion-grouped split + stratified subset

In [ ]:
# Locate dataset (slug/layout can vary) — find HAM10000_metadata.csv under /kaggle/input.
search_roots = [INPUT_DIR] + ([Path("/kaggle/input")] if IN_KAGGLE else [])
meta_csv = None
for root in search_roots:
    if root.exists():
        meta_csv = next(root.rglob("HAM10000_metadata*.csv"), None)
        if meta_csv is not None:
            break
if meta_csv is None:
    raise FileNotFoundError(
        "Could not find HAM10000_metadata.csv. Add the dataset via Add Input -> "
        "'Skin Cancer MNIST: HAM10000' (by kmader).")
INPUT_DIR = meta_csv.parent
print(f"Dataset root: {INPUT_DIR}")

img_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    for p in INPUT_DIR.rglob(ext):
        img_paths.setdefault(p.stem, p)
assert img_paths, f"No image files under {INPUT_DIR}."

with open(meta_csv, newline="") as f:
    rows = list(csv.DictReader(f))

lesion_imgs, lesion_dx = defaultdict(list), {}
for r in rows:
    iid, lid, dx = r["image_id"], r["lesion_id"], r["dx"].lower()
    if iid not in img_paths:
        continue
    lesion_imgs[lid].append(iid)
    lesion_dx[lid] = dx

classes = sorted(DX_TO_CLASS[dx] for dx in set(lesion_dx.values()))
class_to_idx = {c: i for i, c in enumerate(classes)}
assert len(classes) == 7, f"Expected 7 classes, got {classes}"

lesions_by_class = defaultdict(list)
for lid, dx in lesion_dx.items():
    lesions_by_class[DX_TO_CLASS[dx]].append(lid)

# Per-class grouped split: a lesion's images never straddle train/val.
rng = random.Random(SEED)
train_samples, val_samples = [], []
for cls in classes:
    lesions = lesions_by_class[cls]
    rng.shuffle(lesions)
    total = sum(len(lesion_imgs[l]) for l in lesions)
    val_target = min(VAL_CAP, max(1, round(VAL_FRACTION * total)))
    val_n, train_n = 0, 0
    for lid in lesions:
        imgs = [(img_paths[iid], class_to_idx[cls]) for iid in lesion_imgs[lid]]
        if val_n < val_target:
            val_samples.extend(imgs); val_n += len(imgs)
        elif train_n < TRAIN_CAP:
            take = imgs[: max(0, TRAIN_CAP - train_n)]
            train_samples.extend(take); train_n += len(take)

# Stratified subsample of TRAIN: identical fraction from every class, so the class
# prior is unchanged -> the logit-adjustment offset stays valid at full scale.
by_cls = defaultdict(list)
for s in train_samples:
    by_cls[s[1]].append(s)
srng = random.Random(SEED)
subset = []
for ci, items in by_cls.items():
    srng.shuffle(items)
    subset.extend(items[: max(1, round(SUBSET_FRACTION * len(items)))])
srng.shuffle(subset)
train_samples = subset

print(f"Train (subset): {len(train_samples)}   Val: {len(val_samples)}")
for cls in classes:
    ci = class_to_idx[cls]
    tr = sum(1 for _, y in train_samples if y == ci)
    va = sum(1 for _, y in val_samples if y == ci)
    print(f"  {cls:30s} train {tr:5d}  val {va:4d}")


## 4. Data, head-swap, and metric helpers

Head-swap / freeze mirror `skin/skin_model.replace_head`; ECE and the perturbations mirror `skin/04_analysis.py`, so bake-off numbers are comparable to the full analysis.

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, samples, transform, classes):
        self.samples, self.transform, self.classes = samples, transform, classes
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(PROXY_IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_ds = SkinDataset(train_samples, train_tf, classes)
_pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=_pin,
                          persistent_workers=NUM_WORKERS > 0, drop_last=True)


def make_val_loader(perturb=None):
    # Base pipeline -> [0,1] CHW tensor; a perturbation acts on that, then normalise.
    steps = [transforms.Resize(VAL_RESIZE), transforms.CenterCrop(PROXY_IMG_SIZE),
             transforms.ToTensor()]
    if perturb is not None:
        steps.append(transforms.Lambda(perturb))
    steps.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    ds = SkinDataset(val_samples, transforms.Compose(steps), classes)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=_pin)

val_loader = make_val_loader()  # clean, used for epoch selection


def replace_head(model, n_classes):
    for attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, attr, None)
        if head is None:
            continue
        if isinstance(head, nn.Linear):
            setattr(model, attr, nn.Linear(head.in_features, n_classes)); return model
        last = None
        for name, m in head.named_modules():
            if isinstance(m, nn.Linear):
                last = name
        if last is not None:
            parent = head; *path, leaf = last.split(".")
            for p in path:
                parent = getattr(parent, p)
            setattr(parent, leaf, nn.Linear(getattr(parent, leaf).in_features, n_classes))
            return model
    raise ValueError(f"No head on {type(model).__name__}")


def freeze_backbone(model, freeze=True):
    head_ids = set()
    for attr in ["classifier", "head", "heads", "fc"]:
        h = getattr(model, attr, None)
        if h is not None:
            head_ids.update(id(p) for p in h.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_ids) if freeze else True


def head_parameters(model):
    for attr in ["classifier", "head", "heads", "fc"]:
        h = getattr(model, attr, None)
        if h is not None:
            return h.parameters()
    return model.parameters()


def log_class_prior(samples, n_classes):
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    prior = counts / counts.sum()
    return torch.log(torch.tensor(prior, dtype=torch.float32).clamp_min(1e-12))


class LogitAdjustedLoss(nn.Module):
    def __init__(self, log_prior, tau):
        super().__init__(); self.register_buffer("adj", tau * log_prior)
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target)


def normalized_entropy(p):
    p = np.asarray(p, dtype=float); n = len(p)
    return 0.0 if n <= 1 else float(-(p * np.log(p + 1e-12)).sum() / np.log(n))


def expected_calibration_error(probs, targets, n_bins=15):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == targets).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi); c = int(m.sum())
        if c:
            ece += c / len(conf) * abs(conf[m].mean() - acc[m].mean())
    return float(ece)


# Perturbations act on a [0,1] CHW tensor (same defs as 04_analysis.py); we evaluate
# each at its worst severity from the analysis sweep.
def gaussian_noise(std): return lambda t: (t + torch.randn_like(t) * std).clamp(0, 1)
def gaussian_blur(sigma):
    k = 2 * int(np.ceil(2 * sigma)) + 1
    return lambda t: TF.gaussian_blur(t, kernel_size=k, sigma=sigma)
def brightness(factor): return lambda t: TF.adjust_brightness(t, factor).clamp(0, 1)

WORST = {"Darken": brightness(0.25), "Gaussian noise": gaussian_noise(0.30),
         "Blur": gaussian_blur(3.5)}


## 5. The bake-off

In [ ]:
def run_inference(model, loader):
    model.eval(); P, PR, T = [], [], []
    with torch.no_grad(), amp_ctx():
        for x, y in loader:
            logits = model(x.to(device, non_blocking=True))
            p = torch.softmax(logits.float(), 1).cpu().numpy()
            P.append(p); PR.append(p.argmax(1)); T.append(y.numpy())
    return np.concatenate(P), np.concatenate(PR), np.concatenate(T)


def fit_phase(model, crit, opt, scaler, loader, epochs, best, best_state, tag):
    for e in range(epochs):
        model.train()
        for x, y in loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            opt.zero_grad()
            with amp_ctx():
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        _, pr, t = run_inference(model, val_loader)
        bal = balanced_accuracy_score(t, pr)
        print(f"  [{tag} {e + 1}/{epochs}] val bal-acc {bal:.3f}", flush=True)
        if bal > best:
            best = bal
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return best, best_state


def train_one(name):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = replace_head(models.get_model(name, weights="DEFAULT"), len(classes)).to(device)
    crit = LogitAdjustedLoss(log_class_prior(train_samples, len(classes)), LOGIT_ADJUST_TAU).to(device)
    scaler = make_scaler()
    best, best_state = -1.0, None

    freeze_backbone(model, True)
    opt = torch.optim.AdamW(head_parameters(model), lr=HEAD_LR)
    best, best_state = fit_phase(model, crit, opt, scaler, train_loader, HEAD_EPOCHS, best, best_state, "head")

    freeze_backbone(model, False)
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR)
    best, best_state = fit_phase(model, crit, opt, scaler, train_loader, FT_EPOCHS, best, best_state, "ft")

    model.load_state_dict(best_state)
    return model, best


def measure(model):
    probs, preds, targets = run_inference(model, val_loader)
    out = {"bal_acc": float(balanced_accuracy_score(targets, preds)),
           "acc": float((preds == targets).mean()),
           "ece": expected_calibration_error(probs, targets),
           "robust": {}}
    for name, fn in WORST.items():
        p, pr, t = run_inference(model, make_val_loader(fn))
        out["robust"][name] = {
            "bal_acc": float(balanced_accuracy_score(t, pr)),
            "mean_entropy": float(np.mean([normalized_entropy(r) for r in p])),
        }
    return out


results = {}
for name in ARCHITECTURES:
    t0 = time.time()
    print(f"\n=== {name} ===", flush=True)
    model, best = train_one(name)
    m = measure(model)
    m["mins"] = round((time.time() - t0) / 60, 1)
    results[name] = m
    dk = m["robust"]["Darken"]
    print(f"  -> bal-acc {m['bal_acc']:.3f}  acc {m['acc']:.3f}  ECE {m['ece']:.3f}  "
          f"| darken bal-acc {dk['bal_acc']:.3f} ent {dk['mean_entropy']:.3f}  "
          f"({m['mins']} min)", flush=True)
    del model; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


## 6. Results

In [ ]:
order = sorted(results, key=lambda k: -results[k]["bal_acc"])
print(f"{'model':22s}{'bal-acc':>9}{'acc':>7}{'ECE':>7}{'darkenBA':>10}{'darkEnt':>9}{'min':>6}")
print("-" * 70)
for k in order:
    r = results[k]; dk = r["robust"]["Darken"]
    print(f"{k:22s}{r['bal_acc']:>9.3f}{r['acc']:>7.3f}{r['ece']:>7.3f}"
          f"{dk['bal_acc']:>10.3f}{dk['mean_entropy']:>9.3f}{r['mins']:>6.1f}")

# A model that DARKENS confidently-wrong (high bal-acc loss + LOW entropy) is the failure
# we want to avoid. Good = retains accuracy AND/OR raises entropy (stays honestly unsure).
labels = order
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(labels, [results[k]["bal_acc"] for k in labels], color="#6366f1")
axes[0].set_title("Clean balanced accuracy (higher better)")
axes[1].bar(labels, [results[k]["ece"] for k in labels], color="#f59e0b")
axes[1].set_title("ECE (lower better)")
axes[2].bar(labels, [results[k]["robust"]["Darken"]["bal_acc"] for k in labels], color="#10b981")
axes[2].set_title("Darken balanced accuracy (higher better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "bakeoff.png", dpi=130, bbox_inches="tight")
plt.show()

with open(OUT_DIR / "bakeoff_results.json", "w") as f:
    json.dump({"config": {"proxy_img_size": PROXY_IMG_SIZE, "subset_fraction": SUBSET_FRACTION,
                          "head_epochs": HEAD_EPOCHS, "ft_epochs": FT_EPOCHS,
                          "logit_adjust_tau": LOGIT_ADJUST_TAU, "seed": SEED},
               "results": results}, f, indent=2)
print(f"\nWinner (balanced accuracy): {order[0]}")
print(f"Saved -> {OUT_DIR / 'bakeoff_results.json'} and bakeoff.png")
print("Set MODEL_NAME / IMG_SIZE in kaggle_training.ipynb to the winner, then run the full recipe.")
